# FastAPI Backend für unser ERP-System

## Das Szenario

Das unternehmen ist die **Enders Office IT GmbH**, ein Händler für IT-Zubehör und Bürobedarf. Dessen ERP-System muss Pesonen verwalten (Kunden und Lieferanten), Artikel und Belege (Angebote, bestellungen, Rechnungen).





| A | Stammdaten: personen und Artikel | 'stammdaten' |
| B | Einkauf: Lieferanten und Bestellungen | 'einkauf' |
| C | Verkauf: Kunden und Angebote und Rechnungen | 'verkauf' |

### Was wir jetzt machen
1. Das ERP-Schema mit daten aus `information_schema` erkunden
2. FastAPI-Projektstruktur mit einem Router pro gruppe
3. Erste GET-Endpunkte mit Anbindung zur ERP Datenbank
4. Pydantic-Response-Modelle
5. fehlerbehandlung mit HTTPException

In [13]:
from platform import version

## verbindungsparameter
import psycopg2
import psycopg2.errors
import os
from dotenv import load_dotenv

load_dotenv()
verbindung = {
'host': os.getenv('DB_HOST', 'localhost'),
'dbname': os.getenv('DB_NAME', 'erp'),
'user': os.getenv('DB_USER', 'postgres'),
'password': os.getenv('DB_PASSWORD', 'Awb2tz'),
'port': os.getenv('DB_PORT', '5432'),
}

try:
    conn = psycopg2.connect(**verbindung)
    conn.close()
    print("Verbindung zur Datenbank erfolgreich!")
except psycopg2.Error as e:
    print("Fehler bei der Verbindung zur Datenbank:", e)

Verbindung zur Datenbank erfolgreich!


## 1. Das ERP-Schema mit daten aus `information_schema` erkunden
Die ERp-Datenbank hat vir Schemas: `stammdaten`, `einkauf`, `verkauf` und `protokoll`. Bevor man Endpunkte schriebt, schaut man sich die Tabellenstruktur an.
PostgreSQL speichert alle Metadaten in den `information_schema` Tabellen.

In [15]:
# Alle Tabellen der ERP-Datenbank anzeigen
with psycopg2.connect(**verbindung) as conn:
    with conn.cursor() as cur:
        cur.execute("""
        select table_schema, table_name, table_type
        from information_schema.tables
        where table_schema in ('stammdaten', 'einkauf', 'verkauf', 'protokoll')
        order by table_schema, table_name
        """)

        for schema, tabelle, typ in cur.fetchall():
            kennung = "view" if typ == "VIEW" else "table"
            print(f"{schema}.{tabelle} ({kennung})")

einkauf.belege (table)
einkauf.bestelluebersicht (view)
einkauf.positionen (table)
protokoll.aenderungen (table)
protokoll.api_log (table)
stammdaten.adressen (table)
stammdaten.artikel (table)
stammdaten.artikel_bestand (view)
stammdaten.artikelgruppen (table)
stammdaten.belegarten (table)
stammdaten.kunden (view)
stammdaten.laender (table)
stammdaten.lieferanten (view)
stammdaten.personen (table)
stammdaten.zahlungsbedingungen (table)
verkauf.belege (table)
verkauf.offene_rechnungen (view)
verkauf.positionen (table)


In [17]:
# Hilfsfunktionen: Spalten einer Tabelle anzeigen
def spalten_anzeigen(schema: str, tabelle: str):
    with psycopg2.connect(**verbindung) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT column_name, data_type, is_nullable, column_default
                FROM information_schema.columns
                WHERE table_schema = %s AND table_name = %s
                ORDER BY ordinal_position;
            """, (schema, tabelle))
            return cur.fetchall()

# Die wichtigsten Tabellen:
for schema, tabelle in [
    ('stammdaten', 'personen'),
    ('stammdaten', 'artikel'),
    ('einkauf', 'belege'),
    ('verkauf', 'belege'),
]:
    spalten = spalten_anzeigen(schema, tabelle)
    print(f"\n{schema}.{tabelle}")
    for s in spalten:
        # Index auf s[2] geändert, da is_nullable an 3. Stelle selektiert wird
        null_hint = '' if s[1] == 'YES' else ' NOT NULL '
        print(f"  {s[0]:<30} {s[1]}{null_hint}")


stammdaten.personen
  id                             integer NOT NULL 
  typ                            character varying NOT NULL 
  kundennummer                   character varying NOT NULL 
  lieferantennummer              character varying NOT NULL 
  firma                          character varying NOT NULL 
  vorname                        character varying NOT NULL 
  nachname                       character varying NOT NULL 
  email                          character varying NOT NULL 
  telefon                        character varying NOT NULL 
  ust_id                         character varying NOT NULL 
  zahlungsbedingung_id           integer NOT NULL 
  kreditlimit                    numeric NOT NULL 
  aktiv                          boolean NOT NULL 
  notizen                        text NOT NULL 
  angelegt_am                    timestamp without time zone NOT NULL 

stammdaten.artikel
  id                             integer NOT NULL 
  artikelnummer                  cha

In [16]:
# fremdschlüssel anzeigen

with psycopg2.connect(**verbindung) as conn:
    with conn.cursor() as cur:
        cur.execute("""
        select
            tc.table_schema || '.' || tc.table_name as quelle,
            kcu.column_name as spalte,
            ccu.table_schema || ',' || ccu.table_name as ziel
        from information_schema.table_constraints as tc
        join information_schema.key_column_usage as kcu
            on tc.constraint_name = kcu.constraint_name
         join information_schema.constraint_column_usage as ccu
            on ccu.constraint_name = tc.constraint_name
        where tc.constraint_type = 'FOREIGN KEY'
             and tc.table_schema in ('stammdaten', 'einkauf', 'verkauf')
        order by quelle;
        """)

        for quelle, spalte, ziel in cur.fetchall():
            print(f"{quelle}.{spalte} -> {ziel}")

einkauf.belege.belegart -> stammdaten,belegarten
einkauf.belege.belegart -> stammdaten,belegarten
einkauf.belege.lieferant_id -> stammdaten,personen
einkauf.belege.lieferad_id -> stammdaten,adressen
einkauf.belege.lieferad_id -> stammdaten,adressen
einkauf.belege.lieferad_id -> stammdaten,adressen
einkauf.belege.lieferad_id -> stammdaten,adressen
einkauf.belege.zahlungsbedingung_id -> stammdaten,zahlungsbedingungen
einkauf.belege.zahlungsbedingung_id -> stammdaten,zahlungsbedingungen
einkauf.belege.zahlungsbedingung_id -> stammdaten,zahlungsbedingungen
einkauf.belege.vorgaenger_id -> verkauf,belege
einkauf.belege.vorgaenger_id -> verkauf,belege
einkauf.belege.zahlungsbedingung_id -> stammdaten,zahlungsbedingungen
einkauf.belege.belegart -> stammdaten,belegarten
einkauf.belege.belegart -> stammdaten,belegarten
einkauf.belege.vorgaenger_id -> einkauf,belege
einkauf.belege.vorgaenger_id -> einkauf,belege
einkauf.positionen.artikel_id -> stammdaten,artikel
einkauf.positionen.beleg_id -> ve

## Start Fast-API

FastAPi ist ein python-Framwork, das aus normalen python funktionen einen HTTP Server macht. Der wartet dann auf HTTP anfragen und ruft die funktionen auf, um antworten zu generieren. Es ist sehr schnell und einfach zu benutzen, und hat eine tolle integration mit Pydantic, um datenmodelle zu definieren.

fastAPi macht drei dinge automatisch:
1. Es generiert automatisch eine OpenAPI Spezifikation für die API, basierend auf den Endpunkten und den Pydantic Modellen, die man definiert. Das ist eine standardisierte beschreibung der API, die von vielen tools verstanden wird.
2. Es generiert automatisch eine interaktive API-Dokumentation, die auf der OpenAPI Spezifikation basiert
3. Es validiert automatisch die eingehenden Daten gegen die Pydantic Modelle, und gibt automatisch fehler zurück, wenn die daten nicht dem modell entsprechen.

In [20]:
# Das kleinste mögliche FastAPI Programm
import uvicorn
from fastapi import FastAPI, testclient

# FastAPI Objekt anlegen
# Das ist die Anwendung. Sie kennt alle Routen und verwaltet den Server
app = FastAPI(
    title= "Bit connect office IT GmbH - ERP",
    version= "1.3.0",
)

print(type(app))  # <class 'fastapi.applications.FastAPI'>
print(app.title)  # Bit connect office IT GmbH - ERP
print(app.version)  # 1.3.0
print(len(app.routes)) # 0
print(app.openapi()) # {'openapi': '3.0.2', 'info': {'title': 'Bit connect office IT GmbH - ERP', 'version': '1.3.0'}, 'paths': {}

print("zum starten des servers: uvicorn main:app --reload")

uvicorn.run("fast_api:app", host="127.0.0.1", port=8000, reload=True)

INFO:     Will watch for changes in these directories: ['C:\\Users\\CC-Student\\PycharmProjects\\python_learnings\\pythontraining\\fachübergeifend']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [18532] using StatReload


<class 'fastapi.applications.FastAPI'>
Bit connect office IT GmbH - ERP
1.3.0
4
{'openapi': '3.1.0', 'info': {'title': 'Bit connect office IT GmbH - ERP', 'version': '1.3.0'}, 'paths': {}}
zum starten des servers: uvicorn main:app --reload


INFO:     Stopping reloader process [18532]


Hier ist der Text aus deinem Screenshot, sauber formatiert als Markdown, damit du ihn direkt in eine Jupyter-Zelle kopieren kannst:
Was passiert beim Starten mit uvicorn?

uvicorn backend.api:api --reload bedeutet:

    backend.api ist der Python-Modul-Pfad: Ordner backend, Datei api.py.

    :app ist der Name des FastAPI-Objekts in dieser Datei.

    --reload bedeutet hier: bei jeder Änderung automatisch neu starten.

uvicorn ist der Webserver. FastAPI ist das Framework. uvicorn nimmt HTTP-Anfragen entgegen und reicht sie an das FastAPI-Objekt weiter. FastAPI entscheidet dann am Ende, welche Funktion aufgerufen wird und gibt die Antwort zurück.

## Decorators und Routen
Eine **Route** ist eine URL, die auf eine Funktion zeigt. Wenn jemand diese URL aufruft, wird die Funktion ausgeführt und ihre Rückgabe als HTTP-Antwort zurückgegeben.
Ein **Decorator** ist eine spezielle Syntax in Python, die es ermöglicht, Funktionen zu modifizieren oder zu erweitern, ohne ihren Code direkt zu ändern. In FastAPI werden Decorators verwendet, um Routen zu definieren.

In [ ]:
# Beispiel für eine Route mit einem Decorator
@app.get("/personen")
def alle_personen():
    """Diese Funktion wird aufgerufen, wenn jemand die URL /personen mit der GET-Methode aufruft."""
    return {"message": "Hier könnte ihre Werbung stehen."}

Das ist identisch mit:
def alle_personen():
    return []

alle_personen = app.get("/personen")(alle_personen)

FastAPi speichert sich intern : `GET /personen` -> `alle_personen` in einer Tabelle, die es später benutzt, um die Anfragen zu routen.

In [23]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
app = FastAPI()

# erste Route: GET
# GET bedeutet: Daten abrufen, ohne sie zu verändern

@app.get("/")
def startseite():
    # Was dieser Funktion zurück gibt, schickt FastAPI an uns als JSON-Antwort
    # Ein Dictionary wird zu einem JSON-Objekt : {'key', 'value'}
    return {"message": "Willkommen bei der Bit connect Office IT GmbH!", 'status_code': 200, 'version': '1.3.0'}


@app.get("/stammdaten/personen")
def alle_personen():
    return [{'id': 1, 'name': 'Dominik', 'typ': 'kunde'}, {'id': 2, 'name': 'Tominik', 'typ': 'lieferant'}, {'id': 3, 'name': 'Lominik', 'typ': 'kunde'}]

# Testen der API mit dem TestClient
client = TestClient(app)
response = client.get("/stammdaten/personen")
print(response.status_code)  # 200
print(response.json()) # [{'id': 1, 'name': 'Dominik',

for p in response.json():
    print(f"{p['id']}: {p['name']} ({p['typ']})")

200
[{'id': 1, 'name': 'Dominik', 'typ': 'kunde'}, {'id': 2, 'name': 'Tominik', 'typ': 'lieferant'}, {'id': 3, 'name': 'Lominik', 'typ': 'kunde'}]
1: Dominik (kunde)
2: Tominik (lieferant)
3: Lominik (kunde)


## URL-Parameter und QUERY-parameter
Es gibt hier zwei Wege Daten an einen GET-Endpunkt zu übergeben:
1. URL-Parameter: z.B. `/personen/1` -> `1` ist ein URL-Parameter, der in der Route definiert wird
2. Query-Parameter: z.B. `/personen?typ=kunde` -> `typ=kunde` ist ein Query-Parameter, der in der Funktion als Argument definiert wird

In [28]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from typing import Optional

app = FastAPI()

# ---- URL-Parameter ----
@app.get("/stammdaten/personen/{person_id}")
def person_anzeigen(person_id: int): # Hier angepasst!
    return {"id": person_id, "firma!": f'Firma #{person_id}'}


# ---- Query-Parameter ----
# Alles was NICHT in der URL definiert ist, sondern als Argument in der Funktion, ist ein Query-Parameter
@app.get("/stammdaten/personen")
def person_anzeigen(
        typ: Optional[str] = None,   #optional: ?typ=kunde
        nur_aktive: bool = True,    #optional: ?nur_aktive=false (default ist true, also werden nur aktive personen angezeigt)
):
    # Demo daten
    return {
        'filter_type': typ,
        'filter_aktive': nur_aktive,
        'ergebnis': f'Gefiltert nach Typ = {typ}, aktiv = {nur_aktive}',
    }

client = TestClient(app)
print("url parameter")
print(client.get("/stammdaten/personen/1").json())  # {'id': 42, 'firma!': 'Firma #42'}
print("query parameter")
print(client.get("/stammdaten/personen?typ=kunde&nur_aktive=false").json())  # {'filter_type': 'kunde', 'filter_aktive': False, '






url parameter
{'id': 1, 'firma!': 'Firma #1'}
query parameter
{'filter_type': 'kunde', 'filter_aktive': False, 'ergebnis': 'Gefiltert nach Typ = kunde, aktiv = False'}


## Pydantic - Eingaben validieren und Ausgaben strukturieren
pydantic löst gleich zwei probleme:
1. Es validiert die eingehenden Daten, z.B. ob die `person_id` wirklich eine Zahl ist, oder ob die `typ` wirklich ein String ist. Wenn nicht, gibt es automatisch eine Fehlermeldung zurück.
2. Es strukturiert die ausgehenden Daten, z.B. indem es sicherstellt, dass die Antwort immer die gleichen Felder hat, und dass diese Felder die richtigen Datentypen haben. Es kann auch automatisch Dokumentation generieren, basierend auf den Pydantic-Modellen.

In [29]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, field_validator, model_validator
from typing import Optional, Literal
app = FastAPI()

class PersonRespone(BaseModel):
    id: int
    typ: str
    firma: Optional[str] = None
    vorname: Optional[str] = None
    nachname: Optional[str] = None
    email: str
    aktiv: bool


# --- Request-Modell: was beim POST erwartet wird ---
class PersonAnlegen(BaseModel):
    typ: Literal['kunde', 'lieferant', 'beide']  # nur diese Werte sind erlaubt
    firma: Optional[str] = None
    vorname: Optional[str] = None
    nachname: Optional[str] = None
    email: str


    # model_validator: Regel, die mehrere Felder gleichzeitig validiert
    @model_validator(mode='after')
    def firma_oder_name_pflicht(self):
        if not self.firma and not (self.vorname and not self.nachname):
            raise ValueError("Entweder Firma oder Vorname/Nachname müssen angegeben werden.")
        return self
